# Notebook 1.5 

**Author : Ling Thang**

**Date : Oct 7 2025**

## Documentations and Code : 

Purpose : Updating metadata files to create contrast for each disease 

output : Concated MEGA metadata file with both manually currated features (ALL CAPS) and features from original metadata (lower case)

## Key Goal : 

Final metadata for each disease with CONTRAST column in addition to other relevant features

relevant features : columns from GEO metadata that are consitent across all diseases

__________

##### Libraries and Tools


In [1]:
# Python v3.11.12
import pandas as pd 
import os
from typing import Dict, Tuple


##### Initialize paths and files

In [2]:
TB_metadata_path = '../data/RNAseq_data_forDE/metadata'

# read manual labeling
TBlabels =	pd.read_csv("../data/labeling/TB_man.tsv", sep = "\t")

# General EDA

### Ensure column consistency and extract relevant columns

### Get list of Series 

In [3]:
print(TBlabels.shape)
print(TBlabels["GSE_ID"].unique(), TBlabels["GSE_ID"].nunique())
TBlabels.isnull().sum()

(515, 7)
['GSE84076' 'GSE148171' 'GSE198557' 'GSE112482' 'GSE112483' 'GSE132283'
 'GSE143627' 'GSE148731' 'GSE174566' 'GSE193777' 'GSE223863' 'GSE256184'
 'GSE64179' 'GSE64182' 'GSE121049' 'GSE141656' 'GSE143731' 'GSE164287'
 'GSE165708' 'GSE183912' 'GSE211974' 'GSE236156'] 22


GSE_ID              0
GSM_ID              0
CLASSIFICATION      0
SAMPLE_TYPE         0
CELL_SOURCE         0
TISSUE_SOURCE       0
CONTEXT           438
dtype: int64

# CONCAT GEO metadata with manual labeling

creating final metadata data file for each disease (FINAL_METADATA_{DISEASE}.tsv)

##### General Functions
_______

In [4]:
# used in previous interations to identify cols_to_extract -- columns that are informational and columns across all metadata files
def extract_cols_with_counts(path):
	"""
	Extracts all unique column names from files in a directory and counts in how many files each column appears.
	Returns a dictionary {column_name: count}.
	"""
	col_counts = {}
	for filename in os.listdir(path):
		file_path = os.path.join(path, filename)
		if os.path.isfile(file_path):
			df = pd.read_csv(file_path, sep='\t')
			for col in set(df.columns):
				col_counts[col] = col_counts.get(col, 0) + 1
	return col_counts

In [5]:
def create_mega_df_from_dir(metadata_path, labeled_df, cols_of_interest, use_NA_string=False):
    """
    Iterates over all files in Metadata Path, extracts cols_of_interest, 
    keeps only rows with series_id and geo_accession in labeled_df,
    and returns the combined DataFrame.
    Missing columns are filled with 'NA' if use_NA_string is True, otherwise left as NaN.
    """
    mega_df = [] # MEGA DF 
    valid_gse = set(labeled_df['GSE_ID']) # identified Series and that passed manual labeling
    valid_gsm = set(labeled_df['GSM_ID']) # identified Samples and that passed manual labeling

    for fname in os.listdir(metadata_path): # iterate through all files in metadata directory
        if fname.endswith('.tsv'):
            fpath = os.path.join(metadata_path, fname)
            df = pd.read_csv(fpath, sep='\t') # loads as pd DataFrame
            # Ensure all columns of interest are present, fill missing with NaN
            df = df.reindex(columns=cols_of_interest)
            # Filter rows
            if 'series_id' in df.columns and 'geo_accession' in df.columns:
                df = df[df['series_id'].isin(valid_gse) & df['geo_accession'].isin(valid_gsm)] # keep only rows with series_id and geo_accession in labeled_df
                mega_df.append(df) # append to list
    if mega_df: # if list is not empty
        mega_df = pd.concat(mega_df, ignore_index=True)
        if use_NA_string:
            mega_df = mega_df.fillna(pd.NA) # fill missing values with 'NA' string
        return mega_df
    else:
        print("No data found matching criteria.") 
        return pd.DataFrame(columns=cols_of_interest) # return empty if no data found

_______

## Variables and Functions for this section

```python
# defined in the begining of the notebook
TB_metadata_path = '../data/metadata/TB'
```

In [6]:
# Columns of interest to extract from metadata files
cols_to_extract = [
	'series_id',
	'geo_accession',
	'title',
	'source_name_ch1',
	'organism_ch1',
	'characteristics_ch1',
	'characteristics_ch1.1',
	'characteristics_ch1.2',
	'characteristics_ch1.3',
	'molecule_ch1',
	'platform_id',
	'library_source',
	'library_strategy',
	'cell type:ch1',
	'tissue:ch1',
	'time:ch1',
	'time point:ch1',
]

In [7]:
# Final columns to keep in the merged dataframe -- Combined with labeling COLS
FINALCOLS = [
	'title',
	'organism_ch1',
	'GSE_ID', 
	'series_id',
	'GSM_ID',
	'geo_accession',
	'CLASSIFICATION',
	'CLASSIFICATION_STANDARDIZED',
	'SAMPLE_TYPE',
	'source_name_ch1',
	'CELL_SOURCE_CLEAN',
	'CELL_SOURCE',
	'cell type:ch1',
	'CL:CODE',
	'tissue:ch1',
	'TISSUE_SOURCE_CLEAN',
	'TISSUE_SOURCE',
    'UBERON:CODE',
	'CONTEXT',
	'time point:ch1',
	'time:ch1', 
	'characteristics_ch1',
	'characteristics_ch1.1',
	'characteristics_ch1.2',
	'characteristics_ch1.3',
	'molecule_ch1',
	'platform_id',
	'library_source',
	'library_strategy',
	'treatment'
]


______
## TB

In [8]:
TBmeta = create_mega_df_from_dir(TB_metadata_path, TBlabels, cols_to_extract, use_NA_string=True)

In [9]:
TBmeta.head()

,series_id,geo_accession,title,source_name_ch1,organism_ch1,characteristics_ch1,characteristics_ch1.1,characteristics_ch1.2,characteristics_ch1.3,molecule_ch1,platform_id,library_source,library_strategy,cell type:ch1,tissue:ch1,time:ch1,time point:ch1
0,GSE236156,GSM7518958,AM10_AM_Media,bronchoalveolar lavage,Homo sapiens,tissue: bronchoalveolar lavage,cell type: alveolar macrophage,treatment: Media,treatment2: Fresh,total RNA,GPL24676,transcriptomic,RNA-Seq,alveolar macrophage,bronchoalveolar lavage,<NA>,<NA>
1,GSE236156,GSM7518959,AM10_AM_TB,bronchoalveolar lavage,Homo sapiens,tissue: bronchoalveolar lavage,cell type: alveolar macrophage,treatment: M. tuberculosis,treatment2: Fresh,total RNA,GPL24676,transcriptomic,RNA-Seq,alveolar macrophage,bronchoalveolar lavage,<NA>,<NA>
2,GSE236156,GSM7518960,AM10_MDM_Media,blood,Homo sapiens,tissue: blood,cell type: monocyte-derived macrophage,treatment: Media,treatment2: Frozen,total RNA,GPL24676,transcriptomic,RNA-Seq,monocyte-derived macrophage,blood,<NA>,<NA>
3,GSE236156,GSM7518961,AM10_MDM_TB,blood,Homo sapiens,tissue: blood,cell type: monocyte-derived macrophage,treatment: M. tuberculosis,treatment2: Frozen,total RNA,GPL24676,transcriptomic,RNA-Seq,monocyte-derived macrophage,blood,<NA>,<NA>
4,GSE236156,GSM7518962,AM12_AM_Media,bronchoalveolar lavage,Homo sapiens,tissue: bronchoalveolar lavage,cell type: alveolar macrophage,treatment: Media,treatment2: Fresh,total RNA,GPL24676,transcriptomic,RNA-Seq,alveolar macrophage,bronchoalveolar lavage,<NA>,<NA>


In [10]:
TBlabels.head()

,GSE_ID,GSM_ID,CLASSIFICATION,SAMPLE_TYPE,CELL_SOURCE,TISSUE_SOURCE,CONTEXT
0,GSE84076,GSM2226808,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
1,GSE84076,GSM2226809,Disease without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
2,GSE84076,GSM2226817,Disease without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
3,GSE84076,GSM2226825,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
4,GSE84076,GSM2226829,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN


Rows (samples count) are expected to match

In [11]:
def merge_labels_and_metadata(labels_df, mega_df):
	"""
	mega_df - output of create_mega_df_from_dir() function

	Merge labeled metadata and MEGA metadata dataframes on series_id/GSE_ID and geo_accession/GSM_ID.
	Ensures the resulting columns are ordered according to columns_list.
	"""
	merged = pd.merge( # merge on GSE_ID/series_id and GSM_ID/geo_accession
		labels_df, # left 
		mega_df, # right
		left_on=['GSE_ID', 'GSM_ID'], # features on the left dataframe
		right_on=['series_id', 'geo_accession'], # features on the right dataframe
		how='inner'
	)
	# Only keep columns present in FINALCOLS and order them
	merged = merged[[col for col in FINALCOLS if col in merged.columns]]
	# Only return if GSE_ID == series_id and GSM_ID == geo_accession
	if not merged['GSE_ID'].equals(merged['series_id']):
		raise ValueError("GSE_ID and series_id columns do not match exactly.") # prevents GSE inconsistencies
	if not merged['GSM_ID'].equals(merged['geo_accession']):
		raise ValueError("GSM_ID and geo_accession columns do not match exactly.") # prevents GSM inconsistencies
	if merged['GSE_ID'].equals(merged['series_id']) and merged['GSM_ID'].equals(merged['geo_accession']):
		# drop duplicate columns
		merged = merged.drop(columns=['GSE_ID', 'GSM_ID']) # only keep series_id and geo_accession from GEO
		print("Merge successful")
	return merged # only returns if both columns match exactly


In [12]:
TBfinal = merge_labels_and_metadata(TBlabels, TBmeta)
TBfinal.head()

Merge successful


,title,organism_ch1,series_id,geo_accession,CLASSIFICATION,SAMPLE_TYPE,source_name_ch1,CELL_SOURCE,cell type:ch1,tissue:ch1,...,time point:ch1,time:ch1,characteristics_ch1,characteristics_ch1.1,characteristics_ch1.2,characteristics_ch1.3,molecule_ch1,platform_id,library_source,library_strategy
0,Ctrl_Unvac_3,Homo sapiens,GSE84076,GSM2226808,Healthy Control without treatment,blood_sample,Whole blood,whole blood,<NA>,whole blood,...,<NA>,<NA>,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,<NA>,<NA>,polyA RNA,GPL16791,transcriptomic,RNA-Seq
1,ATB_5,Homo sapiens,GSE84076,GSM2226809,Disease without treatment,blood_sample,Whole blood,whole blood,<NA>,whole blood,...,<NA>,<NA>,clinical information: Active Tuberculosis,tissue: whole blood,<NA>,<NA>,polyA RNA,GPL16791,transcriptomic,RNA-Seq
2,ATB_4,Homo sapiens,GSE84076,GSM2226817,Disease without treatment,blood_sample,Whole blood,whole blood,<NA>,whole blood,...,<NA>,<NA>,clinical information: Active Tuberculosis,tissue: whole blood,<NA>,<NA>,polyA RNA,GPL16791,transcriptomic,RNA-Seq
3,Ctrl_Unvac_2,Homo sapiens,GSE84076,GSM2226825,Healthy Control without treatment,blood_sample,Whole blood,whole blood,<NA>,whole blood,...,<NA>,<NA>,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,<NA>,<NA>,polyA RNA,GPL16791,transcriptomic,RNA-Seq
4,Ctrl_Unvac_1,Homo sapiens,GSE84076,GSM2226829,Healthy Control without treatment,blood_sample,Whole blood,whole blood,<NA>,whole blood,...,<NA>,<NA>,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,<NA>,<NA>,polyA RNA,GPL16791,transcriptomic,RNA-Seq


____
## Final Check
____

In [13]:
# Check to make sure that the shapes for labels, meta, and final are the same
print("TB Row Check:", TBlabels.shape[0] == TBmeta.shape[0] == TBfinal.shape[0])

print("\n")

# + 2 to account for dropping GSE_ID and GSM_ID columns
print("TB Column Check:", TBlabels.shape[1] + TBmeta.shape[1] == (TBfinal.shape[1]+2))

TB Row Check: True


TB Column Check: True


all true as we are expecting the rows to stay consistent and columns to add up (excluding the duplicate features)

**Success of merge_signature_and_metadata() function means that Series (GSE_ID) and Sample (GSM_ID) columns match exactly between metadata and signature data for each disease.**

In [14]:
# save final MEGA dfs
# TBfinal.to_csv("../data/FINALMETA/TB_FINAL.tsv", sep = "\t", index = False)

# Make Contrasts

**NOTE : the file above is later used to create `data/labeling/TB_FINAL.tsv` which contains the cells and tissues that were manually mapped to Cell Ontology and UBERON. This file is then used in the next notebook to merge with the final metadata to get the UBERON codes for each sample.**

In [22]:
TBfinal = pd.read_csv("../data/labeling/TB_FINAL.tsv", sep = "\t")
TBfinal.head()

,title,organism_ch1,series_id,geo_accession,CLASSIFICATION_STANDARDIZED,SAMPLE_TYPE,source_name_ch1,CELL_SOURCE_CLEAN,cell type:ch1,CL:CODE,...,time point:ch1,time:ch1,characteristics_ch1,characteristics_ch1.1,characteristics_ch1.2,characteristics_ch1.3,molecule_ch1,platform_id,library_source,library_strategy
0,Ctrl_Unvac_3,Homo sapiens,GSE84076,GSM2226808,Healthy Control without treatment,blood_sample,Whole blood,Blood Cell,NaN,CL:0000081,...,NaN,NaN,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,NaN,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq
1,ATB_5,Homo sapiens,GSE84076,GSM2226809,Disease without treatment,blood_sample,Whole blood,Blood Cell,NaN,CL:0000081,...,NaN,NaN,clinical information: Active Tuberculosis,tissue: whole blood,NaN,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq
2,ATB_4,Homo sapiens,GSE84076,GSM2226817,Disease without treatment,blood_sample,Whole blood,Blood Cell,NaN,CL:0000081,...,NaN,NaN,clinical information: Active Tuberculosis,tissue: whole blood,NaN,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq
3,Ctrl_Unvac_2,Homo sapiens,GSE84076,GSM2226825,Healthy Control without treatment,blood_sample,Whole blood,Blood Cell,NaN,CL:0000081,...,NaN,NaN,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,NaN,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq
4,Ctrl_Unvac_1,Homo sapiens,GSE84076,GSM2226829,Healthy Control without treatment,blood_sample,Whole blood,Blood Cell,NaN,CL:0000081,...,NaN,NaN,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,NaN,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq


## Get Contrast Function

In [25]:
def get_detailed_contrasts(metadata_df: pd.DataFrame, disease_name: str) -> pd.DataFrame:
    """
    Identifies all valid DGE contrasts (>=3 Disease vs >=3 Healthy) and prints 
    the list of contrasts that failed the criteria and why.

    Args:
        metadata_df: The final merged DataFrame for a single disease.
        disease_name: The name of the disease.

    Returns:
        valid_samples_df: DataFrame of samples belonging to valid contrasts.
    """
    # Define required grouping keys and group names
    group_cols = ['series_id', 'CL:CODE', 'UBERON:CODE', 'CONTEXT'] 
    D_name = 'Disease without treatment'
    H_name = 'Healthy Control without treatment'
    MIN_SAMPLES = 3

    metadata_df = metadata_df.copy()
    metadata_df['CONTEXT'] = metadata_df['CONTEXT'].fillna('NoCONTEXT') # Fill NA context for grouping
    
    # Keep only samples designated for DGE comparison
    dge_eligible_df = metadata_df[metadata_df['CLASSIFICATION_STANDARDIZED'].isin([D_name, H_name])]
    
    # Count samples for each unique biological context/potential contrast
    grouped_counts = dge_eligible_df.groupby(group_cols + ['CLASSIFICATION_STANDARDIZED'])['geo_accession'].count().reset_index()
    grouped_counts = grouped_counts.rename(columns={'geo_accession': 'Count'})

    # Pivot to get Disease and Healthy counts side-by-side for comparison
    contrast_counts = grouped_counts.pivot_table(
        index=group_cols, 
        columns='CLASSIFICATION_STANDARDIZED', 
        values='Count', 
        fill_value=0
    ).reset_index()
    
    # Keep contrasts that meet the >= 3 samples in both groups rule
    valid_contrasts = contrast_counts[
        (contrast_counts[D_name] >= MIN_SAMPLES) & 
        (contrast_counts[H_name] >= MIN_SAMPLES)
    ].copy()
    
    # Identify contrasts that failed the rule
    failed_contrasts = contrast_counts[
        ~((contrast_counts[D_name] >= MIN_SAMPLES) & (contrast_counts[H_name] >= MIN_SAMPLES))
    ].copy()
    
    # Print the specific reason for failure for each discarded contrast
    if not failed_contrasts.empty:
        print("\nContrasts That Failed to meet DESEQ Criteria: ")
        for _, row in failed_contrasts.iterrows():
            contrast_name = f"{row['series_id']}_{row['CL:CODE']}_{row['UBERON:CODE']}_{row['CONTEXT']}"
            d_count, h_count = int(row[D_name]), int(row[H_name])
            reason = ""
            
            if d_count < MIN_SAMPLES and h_count < MIN_SAMPLES:
                reason = f"{d_count} Disease & {h_count} Healthy samples"
            elif d_count < MIN_SAMPLES:
                reason = f"{d_count} Disease samples for the contrast"
            elif h_count < MIN_SAMPLES:
                reason = f"{h_count} Healthy samples for the contrast"

            print(f"{contrast_name} - {reason}")

    # Rename count columns and create the unique Contrast ID string
    valid_contrasts.rename(columns={D_name: 'N_Disease', H_name: 'N_Healthy'}, inplace=True)
    valid_contrasts['CONTRAST'] = valid_contrasts.apply(
        lambda row: f"{row['series_id']}_{row['CL:CODE']}_{row['UBERON:CODE']}_{row['CONTEXT']}", axis=1
    )
    
    # Get the key identifiers of the valid contrasts
    valid_keys = [tuple(x) for x in valid_contrasts[group_cols].values]

    # Filter original eligible samples to keep only those in valid contrasts
    valid_samples_df = dge_eligible_df[dge_eligible_df.set_index(group_cols).index.isin(valid_keys)].copy()
    
    # Attach the new CONTRAST ID and final sample counts
    valid_samples_df = pd.merge(
        valid_samples_df, 
        valid_contrasts[group_cols + ['CONTRAST', 'N_Disease', 'N_Healthy']], 
        on=group_cols, 
        how='left'
    )

    # Define the path for the expression matrix based on series_id
    valid_samples_df['EXPRMAT_PATH'] = valid_samples_df['series_id'].apply(
        lambda x: f"data/FinalExpression/{disease_name}/{x}.tsv"
    )

    # rename CLASSIFICATION_STANDARDIZED back to CLASSIFICATION for output
    valid_samples_df.rename(columns={'CLASSIFICATION_STANDARDIZED': 'CLASSIFICATION'}, inplace=True)

    # Select and reorder final columns for output
    final_cols_order = [
        'series_id', 'geo_accession', 'CLASSIFICATION', 
        'CONTRAST', 'EXPRMAT_PATH'
    ]
    
    return valid_samples_df[final_cols_order].reset_index(drop=True)

## TB

In [26]:
TB_DGE_df = get_detailed_contrasts(TBfinal, 'TB')

## Check the outputs

In [28]:
# standardize the contrast columns delete : and spaces
TB_DGE_df['CONTRAST'] = TB_DGE_df['CONTRAST'].str.replace(':', '', regex=False).str.replace(' ', '', regex=False)

In [32]:
TB_DGE_df

,series_id,geo_accession,CLASSIFICATION,CONTRAST,EXPRMAT_PATH
0,GSE84076,GSM2226808,Healthy Control without treatment,GSE84076_CL0000081_UBERON0000178_NoCONTEXT,data/FinalExpression/TB/GSE84076.tsv
1,GSE84076,GSM2226809,Disease without treatment,GSE84076_CL0000081_UBERON0000178_NoCONTEXT,data/FinalExpression/TB/GSE84076.tsv
2,GSE84076,GSM2226817,Disease without treatment,GSE84076_CL0000081_UBERON0000178_NoCONTEXT,data/FinalExpression/TB/GSE84076.tsv
3,GSE84076,GSM2226825,Healthy Control without treatment,GSE84076_CL0000081_UBERON0000178_NoCONTEXT,data/FinalExpression/TB/GSE84076.tsv
4,GSE84076,GSM2226829,Healthy Control without treatment,GSE84076_CL0000081_UBERON0000178_NoCONTEXT,data/FinalExpression/TB/GSE84076.tsv
...,...,...,...,...,...
510,GSE236156,GSM7519008,Disease without treatment,GSE236156_BALF_BALF_NoCONTEXT,data/FinalExpression/TB/GSE236156.tsv
511,GSE236156,GSM7519009,Healthy Control without treatment,GSE236156_BALF_BALF_NoCONTEXT,data/FinalExpression/TB/GSE236156.tsv
512,GSE236156,GSM7519010,Healthy Control without treatment,GSE236156_BALF_BALF_NoCONTEXT,data/FinalExpression/TB/GSE236156.tsv
513,GSE236156,GSM7519011,Disease without treatment,GSE236156_BALF_BALF_NoCONTEXT,data/FinalExpression/TB/GSE236156.tsv


In [30]:
# save all DGE dfs
# TB_DGE_df.to_csv("../data/labeling/TB_DGE_inputs.tsv", sep = "\t", index = False)

In [31]:
# Check to make sure that the shapes for labels, meta, and final are the same
print(f"TB samples loss check :", TB_DGE_df.shape[0] - TBfinal.shape[0])

TB samples loss check : 0
